# Session 6 · **Meet LangChain**
## Rebuilding the DilSeVector Concierge — from first principles to a working RAG pipeline

---

### The story so far

**Session 3** — Prasad found her. A pink café in Dadar, roses and green chairs. We learned how a vector search narrows a city down to one person.

**Session 4** — That became an app for a hundred million people. Quantization, hybrid search, reranking, and a p99 that doesn't ruin anyone's evening.

**Session 5** — The app learned to talk. We built the Concierge by hand: a search function, a prompt, a memory list, a router, and four checks that catch a lying answer.

**Today** — Prasad has news. The investors love the Concierge so much they want it for three more apps by the end of next month.

There's a problem. Our Session 5 code is a few hundred lines that only we understand. Every piece is hand-wired to every other piece. Want to try a different model? Rewrite the calling code. A different vector database? Rewrite the search. A different embedding model? Rewrite half of it.

So today we rebuild it on **LangChain**.

---

## 0.1 · What is LangChain — and why it's the answer to Prasad's problem

**LangChain is a toolkit for building applications on top of language models.**

That sentence is doing a lot of work, so let's unpack it with the code Prasad's problem actually lives in.

In Session 5 our Concierge did five things:

| Step | What we used | What we wrote ourselves |
|---|---|---|
| Turn a question into numbers | `sentence-transformers` | the call, the loop |
| Find matching pieces | `pinecone` | the query, unpacking results |
| Build a prompt | plain Python | an f-string with slots |
| Ask the model | `google.generativeai` | the call, the config |
| Get the text out | plain Python | `reply.text` |

Five libraries, five different calling styles. **Our code was the glue.** And glue is brittle:

- Swap Gemini for Claude → every model call changes
- Swap Pinecone for Chroma → the search function is rewritten
- Want answers to stream word-by-word → write it yourself, per provider
- Want to see what the prompt actually looked like → print statements everywhere

Now multiply that by three more apps and a one-month deadline. One might use Claude instead of Gemini. One might need a different vector database. All three need streaming. Hand-wiring each one from scratch is exactly the trap.

LangChain's proposal: **make every piece the same shape, so the glue disappears.**

---

## 0.2 · What LangChain is *not*

Worth saying early, because the name misleads people:

- ❌ It is **not a model.** It doesn't answer anything. It calls models you choose.
- ❌ It is **not a database.** It talks to vector databases you set up.
- ❌ It is **not required.** Session 5 worked fine without it.
- ✅ It **is** a set of standard shapes, plus a large catalogue of adapters that fit those shapes.

> **When is it worth it?** When you're building more than one thing, or when you expect to swap parts. Prasad needs the Concierge for three more apps — that's exactly the case for a framework. For a single throwaway script, the plain version is often better.

---

## 0.3 · The ecosystem: which package holds what

LangChain isn't one package. Installing the wrong one, or following a tutorial written for an older version, is the single most common way beginners get stuck.

| Package | Holds | Stability |
|---|---|---|
| `langchain-core` | The base shapes: Runnable, Document, messages, prompts, parsers | 🟢 very stable — the spine |
| `langchain` | The modern top layer, agents, `init_chat_model` | 🟢 current |
| `langchain-classic` | Older chains and retrievers moved out of `langchain` in v1 | 🟡 still supported |
| `langchain-google-genai`, `langchain-openai`, `langchain-pinecone`, … | One package per provider | 🟢 versioned separately |
| `langchain-community` | The old catch-all for everything | 🔴 being retired |
| `langgraph` | Stateful apps, agents, durable memory | 🟢 current |

### ⚠️ The version trap — read this before you copy code from the internet

LangChain released **version 1**, and it moved things. On a clean install today:

```python
from langchain.retrievers import EnsembleRetriever    # ❌ ModuleNotFoundError
```

The whole `langchain.retrievers` module is gone — it's `langchain_classic.retrievers` now. Most blog posts and Stack Overflow answers predate this.

**This notebook pins versions so everything works.** If you install LangChain yourself without those pins, expect import errors.

> **The lesson underneath the annoyance:** a framework's API is *rented*. The concepts underneath — embed, retrieve, prompt, generate — are *owned*. You learned the concepts in Sessions 3–5, which is why a rename can't strand you.


---
## 0.4 · What you'll actually learn today

Most LangChain tutorials show you twenty components and hope you work out the pattern. We're going to do it the other way round.

LangChain has **one idea**. Everything else is that idea, repeated.

We'll learn the idea first — with tiny examples, no API keys, nothing that can go wrong — and only then meet the real components. By the end you'll be able to read LangChain code you've never seen before, because you'll recognise the shape.

## 0.5 · The building blocks

Almost everything in LangChain falls into eight families. You'll meet all eight today.

| # | Family | Job | Today |
|---|---|---|---|
| 1 | **Models** | Talk to an LLM | Part 3 |
| 2 | **Prompts** | Build the input from a template | Part 4 |
| 3 | **Output parsers** | Turn the reply into something usable | Part 5 |
| 4 | **Documents & loaders** | Carry text plus metadata | Part 6 |
| 5 | **Splitters** | Cut long text into pieces | Part 6 |
| 6 | **Embeddings & vector stores** | Turn text into numbers, store and search | Part 7 |
| 7 | **Retrievers** | Given a question, return documents | Parts 8–9 |
| 8 | **Chains (LCEL)** | Join all of the above | Parts 2, 10 |

And one idea holds all eight together. That's Part 1.

### The skeleton every RAG shares

Memory RAG, Branched RAG, Adaptive RAG, Corrective RAG — the differences are all layers on one
skeleton. Today rebuilds each bone of it in LangChain.

| Step | Session 5, by hand | Today |
|---|---|---|
| Ingest and store | BeautifulSoup, embed in a loop, upsert | Documents, splitters, embeddings |
| Transform the query | embed it raw | multi-query, self-query, cheap fixes |
| Retrieve candidates | our `search()` | a retriever, and `alpha` |
| Prepare the context | slice the top few | compress, dedup, reorder, budget |
| Build the prompt | one f-string | `ChatPromptTemplate`, roles, few-shot |
| Generate | call the model, take `.text` | `prompt \| llm \| parser`, streaming free |
| Check it | **our four checks** | **still ours — Part 14** |

### Today's plan

| Part | Topic | Needs API keys? |
|---|---|---|
| 1 | The one idea: Runnables | ❌ no |
| 2 | The pipe `\|` and LCEL | ❌ no |
| 3 | Models and messages | ✅ |
| 4 | Prompts, system vs human, few-shot | ✅ |
| 5 | Output parsers, including JSON | ✅ |
| 6 | Documents, loaders, splitters | ❌ no |
| 7 | Embeddings and vector stores | ✅ |
| 8 | Retrievers and the alpha knob | ✅ |
| 9 | Smarter retrieval: multi-query, self-query, compression, diversity, budget | ✅ |
| 10 | Rebuilding the Concierge | ✅ |
| 11 | Conversation memory | ✅ |
| 12 | Self-Adaptive RAG: letting the question pick the settings | ✅ |
| 13 | Seeing inside your chain | ✅ |
| 14 | What LangChain gave us, and what it left us | — |
| A–B | Migrating old code · when something breaks | — |

### 📘 About the code
Same rule as Session 5: **`for` loops, `if` statements, lists, dictionaries, f-strings.** Every step gets its own line and its own name. No lambdas, no comprehensions, no classes.

LangChain code online is often written very compactly. We'll write it readably first, then look at the compact form — so you can *read* other people's code without having to *write* like that.


## 🗺️ Table of contents

| Part | Topic | Needs API keys? |
|---|---|---|
| 1 | The one idea: Runnables | ❌ no |
| 2 | The pipe `\|` and LCEL | ❌ no |
| 3 | Models and messages | ✅ |
| 4 | Prompts, system vs human, few-shot | ✅ |
| 5 | Output parsers, including JSON | ✅ |
| 6 | Documents, loaders, splitters | ❌ no |
| 7 | Embeddings and vector stores | ✅ |
| 8 | Retrievers and the alpha knob | ✅ |
| 9 | Smarter retrieval: multi-query, self-query, compression, diversity, budget | ✅ |
| 10 | Rebuilding the Concierge (the full RAG chain) | ✅ |
| 11 | Conversation memory | ✅ |
| 12 | Self-Adaptive RAG | ✅ |
| 13 | Seeing inside your chain (debugging) | ✅ |
| 14 | What LangChain gave us, and what it left us | — |
| A | Appendix — migrating old LangChain code | — |
| B | Appendix — when something breaks | — |

Run the cells **top to bottom, once each** — later parts reuse objects defined
earlier (`llm`, `embeddings`, `retriever`, `chunks`, `concierge_chain`, `history`, and so on).

**Before running Part 3 onwards**, add your two secrets in Colab's 🔑 **Secrets**
panel (left sidebar) — `GOOGLE_API_KEY` and `PINECONE_API_KEY` — and enable
notebook access for both. (If you'd rather keep using key files, the setup
cell below falls back to reading `gemini_key2.txt` / `api_key.txt` if the
Secrets aren't found.)

You'll also need the `coffee_pages/` folder of source HTML files, and a
Pinecone index named **`coffee-hybrid`** created for **hybrid search**
(dotproduct metric) matching the `text-embedding-004` embedding dimension.

---
# PART 1 · The one idea: **everything is a Runnable**

If you remember one thing from today, remember this:

> **A Runnable is anything with an `.invoke()` method.**
> Give `.invoke()` an input, get an output back.

A prompt is a Runnable. A model is a Runnable. A retriever is a Runnable. A plain Python function can be made into one.

Because they all share that shape, they all fit together — and every one of them gets `.invoke()`, `.batch()` and `.stream()` without you writing anything.

Let's prove it with the smallest example possible. **No API key needed for Parts 1, 2 and 6.**

In [ ]:
!pip -q install "langchain>=1.0" langchain-classic langchain-core langchain-text-splitters langchain-google-genai langchain-pinecone langchain-community pinecone pinecone-text beautifulsoup4 lark

from langchain_core.runnables import RunnableLambda

# an ordinary Python function
def shout(text):
    return text.upper()

# wrap it, and it becomes a Runnable
shout_step = RunnableLambda(shout)

print("invoke:", shout_step.invoke("turmeric coffee"))

### That's a lot of ceremony for `.upper()`

For one function, yes. Watch what the wrapper bought us though.

Plain `shout()` does exactly one thing. `shout_step` does three, and we wrote no extra code:

In [ ]:
print("one at a time :", shout_step.invoke("turmeric coffee"))
print("a whole list  :", shout_step.batch(["turmeric coffee", "filter coffee", "cold brew"]))

print("piece by piece:", end=" ")
for piece in shout_step.stream("turmeric coffee"):
    print(piece, end="")
print()

---
# PART 2 · The pipe `|` and LCEL

If two things both have `.invoke()`, and the first one's output fits the second one's input, you can join them with `|`.

Read `|` as **"and then send the result into"**.

```
step_one  |  step_two
    ↓            ↓
  runs      receives step_one's output
```

Joining Runnables like this is called **LCEL** — LangChain Expression Language. It's a grand name for a simple idea: same-shaped things clip together.

In [ ]:
def add_excitement(text):
    return text + "!!!"

excite_step = RunnableLambda(add_excitement)

pipeline = shout_step | excite_step

print("shout only :", shout_step.invoke("turmeric coffee"))
print("both steps :", pipeline.invoke("turmeric coffee"))

### The joined thing is *also* a Runnable

This is what makes the design work. `pipeline` isn't a special "chain object" — it's just another Runnable. It has `.invoke()`, `.batch()`, `.stream()`, and you can pipe it into something else.

Small pieces → same shape → join into bigger pieces → still the same shape.

In [ ]:
print("the joined pipeline batches too:")
print(pipeline.batch(["turmeric coffee", "cold brew"]))

def make_question(text):
    return "Have you tried " + text.lower() + "?"

longer = pipeline | RunnableLambda(make_question)
print("\nthree steps:", longer.invoke("turmeric coffee"))

### `RunnablePassthrough` — the "leave it alone" step

Sometimes a chain needs a value passed through untouched. That's `RunnablePassthrough()`. It looks pointless on its own; it becomes essential in Part 10, when a chain has to carry the original question forward while another branch does the searching.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

keep = RunnablePassthrough()
print("passthrough gives back exactly what it got:", keep.invoke("turmeric coffee"))

---
# PART 3 · Models and messages

Time for real keys.

### Two things to understand about chat models

**1. They take a list of messages, not a string.** Each message has a role:

| Role | Class | Holds |
|---|---|---|
| system | `SystemMessage` | your rules and instructions |
| human | `HumanMessage` | what the user typed |
| ai | `AIMessage` | what the model said before |

This isn't decoration. Chat models are **trained** to treat system instructions as more authoritative than user text. Rules in the system slot are followed more reliably than the same rules pasted into the user's message.

**2. They give back a message object, not a string.** The text sits in `.content`, alongside token counts and other details.

In [ ]:
import os

# --- Load secrets: prefer Colab's Secrets panel, fall back to local key files ---
# This mirrors Session 5's approach (Colab userdata) so the same two secrets work
# everywhere. If you already have gemini_key2.txt / api_key.txt uploaded, those
# still work as a fallback.
try:
    from google.colab import userdata
    gemini_key = userdata.get("GOOGLE_API_KEY")
    pinecone_key = userdata.get("PINECONE_API_KEY")
except Exception:
    gemini_key = None
    pinecone_key = None

if not gemini_key and os.path.exists("gemini_key2.txt"):
    gemini_key = open("gemini_key2.txt").read().strip()
if not pinecone_key and os.path.exists("api_key.txt"):
    pinecone_key = open("api_key.txt").read().strip()

if not gemini_key:
    raise RuntimeError(
        "GOOGLE_API_KEY not found. Add it via Colab -> Secrets, or upload gemini_key2.txt."
    )
if not pinecone_key:
    raise RuntimeError(
        "PINECONE_API_KEY not found. Add it via Colab -> Secrets, or upload api_key.txt."
    )

os.environ["GOOGLE_API_KEY"] = gemini_key
os.environ["PINECONE_API_KEY"] = pinecone_key

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# init_chat_model is LangChain's one constructor for ANY provider: swap the
# "google_genai:..." string for "anthropic:..." or "openai:..." and the rest
# of this notebook does not change.
llm = init_chat_model("google_genai:gemini-2.0-flash", temperature=0)

conversation = [
    SystemMessage(content="You are a coffee expert. Answer in one short sentence."),
    HumanMessage(content="What is masala coffee?")
]

reply = llm.invoke(conversation)

print("type of reply:", type(reply).__name__)
print("the text     :", reply.content)

### One constructor, any provider

Look at that string: `"google_genai:gemini-2.0-flash"`. It's `provider:model`.

To switch to Claude you change **that one string** to `"anthropic:claude-sonnet-4-5"` and install that provider package. Nothing else in this notebook changes.

That is the swappability we paid for. It's the clearest single example of what a framework buys you.

> **On `temperature`:** 0 means "always pick the most likely next word" — repeatable, best for RAG where you want the facts from the context, not invention. Higher values add randomness, which is good for creative writing and bad for us.

---
# PART 4 · Prompts

In Session 5 our prompt was one long f-string. A `ChatPromptTemplate` is better in two ways: it keeps roles separate, and the blanks are named slots you fill at run time.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a coffee expert. Answer in one short sentence."),
    ("human", "What is {drink}?")
])

filled = prompt.invoke({"drink": "masala coffee"})

print("prompt.invoke gives back messages:")
for message in filled.messages:
    print("  ", type(message).__name__, "->", message.content)

### A prompt is a Runnable too

So it pipes. Here is the smallest real chain in LangChain:

```
prompt  |  llm
```

🎲 **Before you run it:** what type will come out — a string, or a message object?

In [ ]:
small_chain = prompt | llm
result = small_chain.invoke({"drink": "filter coffee"})

print("type:", type(result).__name__)
print("text:", result.content)

### Showing the model examples (few-shot)

Sometimes describing the format you want isn't enough — showing is better. A few-shot prompt inserts example exchanges *before* the real question, so the model copies the pattern.

Useful when you need a strict output style, like always attaching a citation.

> **The catch worth knowing:** examples teach format, but they also teach bias. If all your examples answer "no", the model leans toward "no". Choose examples that vary in their conclusions.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

example_shape = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])

examples = [
    {"question": "Is turmeric coffee caffeinated?",
     "answer": "No, it is caffeine free when made with milk alone [03_turmeric]."},
    {"question": "Is filter coffee strong?",
     "answer": "Yes, the chicory blend gives it a strong body [04_filter]."}
]

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_shape,
    examples=examples
)

styled_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence and always end with a [page] citation."),
    few_shot,
    ("human", "{question}")
])

built = styled_prompt.invoke({"question": "Is masala coffee sweet?"})
print("messages sent to the model:")
for message in built.messages:
    print("  ", type(message).__name__, "->", message.content[:70])

---
# PART 5 · Output parsers

The model returns a message object. A parser turns it into something your program can use. This is the `reply.text` we wrote by hand in Session 5 — now it's a Runnable, so it joins the chain instead of sitting outside it.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

full_chain = prompt | llm | parser
answer = full_chain.invoke({"drink": "cold brew"})

print("type:", type(answer).__name__)
print("text:", answer)

### Getting JSON back

Sometimes you want structured data, not prose — for example a router that decides which kind of question was asked. `JsonOutputParser` asks for JSON and gives you a real Python dictionary.

This is Session 5's branching router, rebuilt.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Decide what kind of coffee question this is.\n"
     'Reply with ONLY JSON like: {{"branch": "compare", "topics": ["turmeric", "masala"]}}\n'
     "branch must be one of: explain, compare, recommend, lookup."),
    ("human", "{question}")
])

router_chain = router_prompt | llm | JsonOutputParser()

for question in ["Compare turmeric coffee with masala coffee",
                 "How much water for cold brew?"]:
    decision = router_chain.invoke({"question": question})
    print(decision, " <-", question)
    print("   branch is a real Python value:", decision["branch"])

---
# PART 6 · Documents, loaders and splitters

**No API key needed for this part.**

A **Document** is a small box with two compartments:

- `page_content` — the text
- `metadata` — a dictionary of everything else

Every LangChain tool that handles text speaks `Document`. It's the common currency.

> **Why metadata matters more than it looks.** In Session 5 we tracked page ids in a parallel dictionary and kept it in sync by hand. Here the id travels *inside* the document. That's what makes citations survive — no matter how many tools reorder, filter or shorten the text, `metadata["page"]` comes along.

In [ ]:
from langchain_core.documents import Document

example = Document(
    page_content="Turmeric coffee is usually caffeine free when made with milk alone.",
    metadata={"page": "03_turmeric_coffee_haldi_cappuccino", "section": "Overview"}
)

print("text    :", example.page_content)
print("metadata:", example.metadata)
print("page id :", example.metadata["page"])

### Splitters: why we cut documents up

A whole page is too big to embed as one vector. The vector becomes an *average* of everything on the page, so it matches everything a bit and nothing sharply.

`RecursiveCharacterTextSplitter` is the standard tool. The word "recursive" describes how it chooses where to cut: it tries the separators **in order**, and only falls to the next one if the piece is still too big.

```
"\n\n"  try paragraph breaks first
"\n"    then line breaks
". "    then sentence ends
" "     then spaces
""      then give up and cut mid-word
```

That ordering is why your paragraphs usually survive.

**Two settings you must choose — there are no correct defaults:**

| Setting | Too small | Too big |
|---|---|---|
| `chunk_size` | sentences lose their context | vector averages out, matches nothing sharply |
| `chunk_overlap` | a fact split across the boundary is lost | wasted storage and duplicate results |

Overlap is usually 10–20% of chunk size.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = ("Masala coffee is brewed with spices. " * 20).strip()
big_document = Document(page_content=long_text, metadata={"page": "demo_page"})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

pieces = splitter.split_documents([big_document])

print("1 document became", len(pieces), "pieces")
for piece in pieces[:3]:
    print("  ", len(piece.page_content), "chars |", piece.metadata)

### 🔑 Metadata is copied to every piece

Look at the output above: every piece kept `page: demo_page`.

That is what lets you chunk aggressively and still cite by page. Ten chunks from one page all carry the same page id, so a citation still points somewhere a human can open.

### Our coffee pages: splitting by section instead

Here's a judgement call the framework can't make for you.

Our pages all have the same eight `<h2>` sections — Overview, Ingredients, Method, and so on. So we don't need to *guess* at boundaries with a character count. **We already know where the natural boundaries are.**

> **The rule:** `RecursiveCharacterTextSplitter` is for text with no structure. If your documents have structure, use it. Ours do.

In [ ]:
import os
import glob
from bs4 import BeautifulSoup


def read_one_page(filename):
    html = open(filename, encoding="utf-8", errors="ignore").read()
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "head"]):
        tag.decompose()

    title = soup.find("h1").get_text(strip=True)
    page_id = os.path.basename(filename).replace(".html", "")

    documents = []
    for heading in soup.find_all("h2"):
        section_name = heading.get_text(strip=True)

        section_text = ""
        for item in heading.next_siblings:
            if getattr(item, "name", None) == "h2":
                break
            if hasattr(item, "get_text"):
                section_text = section_text + " " + item.get_text(" ", strip=True)
        section_text = " ".join(section_text.split())

        if len(section_text) > 40:
            info = {"page": page_id, "title": title, "section": section_name}
            full_text = title + " - " + section_name + ". " + section_text
            documents.append(Document(page_content=full_text, metadata=info))

    return documents


chunks = []
for filename in sorted(glob.glob("coffee_pages/*.html")):
    for document in read_one_page(filename):
        chunks.append(document)

print("Documents made:", len(chunks))
print("Example:", chunks[0].metadata)

---
# PART 7 · Embeddings and vector stores

**Embeddings** turn text into a list of numbers. Two texts with similar meaning get similar numbers.

Two rules from Session 3, and they still apply:

1. **The same model must embed your documents and your questions.** Different models produce numbers that aren't comparable.
2. **Changing the embedding model means re-indexing everything.** There is no shortcut.

An embeddings object has two methods, because some models treat a question differently from a document:
- `embed_documents(list_of_texts)`
- `embed_query(one_question)`

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vector = embeddings.embed_query("is turmeric coffee caffeine free")

print("a question became", len(vector), "numbers")
print("first five:", vector[:5])

---
# PART 8 · Retrievers

> **A retriever is a Runnable that takes a question (a string) and returns a list of Documents.**

That's the whole contract. It's why retrievers pipe, and why a reranker wrapped around a retriever is *itself* a retriever.

Our index is a **hybrid** index: every piece is stored twice, once as a "meaning" vector and once as an "exact words" vector. The `alpha` setting decides how much each side counts.

In [ ]:
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index("coffee-hybrid")

texts = []
for chunk in chunks:
    texts.append(chunk.page_content)

bm25 = BM25Encoder().default()
bm25.fit(texts)

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    top_k=5,
    alpha=0.5
)

UPLOAD = False
if UPLOAD:
    metadatas = []
    for chunk in chunks:
        metadatas.append(chunk.metadata)
    retriever.add_texts(texts=texts, metadatas=metadatas)

found = retriever.invoke("is turmeric coffee caffeine free")

print("Returned", len(found), "Documents:")
for document in found[:3]:
    print("  ", document.metadata["page"][:36], "|", document.metadata["section"])

### ⚠️ The alpha knob — and why you should never trust a comment

```
alpha = 1.0   →  meaning only      (dense)
alpha = 0.0   →  exact words only  (sparse / BM25)
alpha = 0.5   →  both
```

**Alpha turns up the "meaning" side.** Same α as Session 4: α = 1 was the Romantic, α = 0 was the Matchmaking Aunty.

Plenty of tutorials state this backwards. So don't take my word for it either — ask the library.

In [ ]:
from pinecone_text.hybrid import hybrid_convex_scale
import inspect

print(inspect.getdoc(hybrid_convex_scale))

dense_part, sparse_part = hybrid_convex_scale([1.0, 1.0], {"indices": [0], "values": [1.0]}, alpha=1.0)
print("\nalpha = 1.0 -> dense", dense_part, "sparse", sparse_part["values"], " => MEANING ONLY")

dense_part, sparse_part = hybrid_convex_scale([1.0, 1.0], {"indices": [0], "values": [1.0]}, alpha=0.0)
print("alpha = 0.0 -> dense", dense_part, "sparse", sparse_part["values"], " => EXACT WORDS ONLY")

### Turn the knob and watch the results move
🎲 **Before you run it:** "ashwagandha" is an exact word on one page. Which alpha finds that page most reliably?

In [ ]:
question = "ashwagandha for stress"

for alpha_value in [0.0, 0.5, 1.0]:
    retriever.alpha = alpha_value

    if alpha_value == 0.0:
        label = "exact words only"
    elif alpha_value == 1.0:
        label = "meaning only"
    else:
        label = "both"

    print("\nalpha =", alpha_value, "(" + label + ")")
    for document in retriever.invoke(question)[:3]:
        print("   ", document.metadata["page"][:36], "|", document.metadata["section"])

retriever.alpha = 0.5

> **Theory: what the framework took, and what it kept.**
> It took the *plumbing* — calling the embedder, querying the index, mixing two score lists. It did not take the *decision*. Alpha still has no correct default, and only you know whether your users type exact product codes or vague descriptions.
>
> That pattern repeats for every setting today: `top_k`, `chunk_size`, which embedding model, every word of the prompt.


---
# PART 9 · Smarter retrieval

Ready-made retrievers and post-processors. Each is a few lines, and each fixes a different failure.

| Section | Fixes |
|---|---|
| 9.1 multi-query & decomposition | the user's words don't match the page's words |
| 9.2 self-query | the question implies a filter nobody wrote |
| 9.3 compression | the right sentence is buried in a long passage |
| 9.4 dedup & diversity | ten results that all say the same thing |
| 9.5 cheap transforms | typos and other languages, without an LLM call |
| 9.6 budget | more context than the window can take |

## 9.1 · Multi-query — when the user's words don't match the page's words

The user's phrasing is one sample out of many possible phrasings. Searching once is one roll of the dice.

`MultiQueryRetriever` asks the model to rewrite the question several ways, searches with all of them, and merges the results.

**Cost:** one extra LLM call. **Buys:** recall.

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_retriever = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)

results = multi_retriever.invoke("what is that golden coloured drink and does it keep you awake")

pages_found = []
for document in results:
    page = document.metadata["page"]
    if page not in pages_found:
        pages_found.append(page)

print("Pages found by searching several wordings:")
for page in pages_found:
    print("  ", page)

### The same class, a different job: sub-question decomposition

`MultiQueryRetriever` isn't limited to paraphrasing — swap the prompt and it will **split** a
multi-part question into parts, then search for each.

Paraphrasing helps when the user's words don't match the page's words. Decomposition helps when
one question is secretly three.

In [ ]:
from langchain_core.prompts import PromptTemplate

decomp_prompt = PromptTemplate.from_template(
    "Decompose into 2-4 concise sub-questions that together answer: {question}\n"
    "Return one per line."
)

decomposer = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm,
    prompt=decomp_prompt,
    include_original=True
)

results = decomposer.invoke(
    "Compare turmeric coffee and saffron coffee for taste and health effects"
)

pages_found = []
for document in results:
    page = document.metadata["page"]
    if page not in pages_found:
        pages_found.append(page)

print("pages found by decomposing the question:")
for page in pages_found:
    print("   ", page)

## 9.2 · Self-query — letting the model write the filter

In Session 5 we wrote a router by hand to decide when to filter by section. `SelfQueryRetriever` does it from a description of your metadata: it splits the sentence into a **meaning part** (for the vector search) and a **filter part** (for the database).

"How do I make the saffron one" should become: search for *saffron*, filter to `section = Method`.

**Failure to know about:** if the model invents a filter for a value that doesn't exist, you get zero results. Always keep a fallback.

In [ ]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_pinecone import PineconeVectorStore

plain_store = PineconeVectorStore(index=index, embedding=embeddings, text_key="text")

field_info = [
    AttributeInfo(name="section", type="string",
                  description="One of: Overview, Ingredients (Base & Optional), Method, "
                              "Variations & Regional Twists, Flavor Profile & Pairings, "
                              "Benefits, Cautions & Practical Tips, Cultural & Historical Notes, FAQ"),
    AttributeInfo(name="page", type="string", description="Page id, e.g. 10_saffron_coffee_kesar_coffee"),
    AttributeInfo(name="title", type="string", description="Page title")
]

self_query = SelfQueryRetriever.from_llm(
    llm, plain_store, "Indian coffee pages, split into sections", field_info, verbose=True)

try:
    results = self_query.invoke("how do I make the saffron one")
    for document in results[:3]:
        print("  ", document.metadata.get("page", "?")[:34], "|", document.metadata.get("section"))
except Exception as problem:
    print("Self-query failed:", type(problem).__name__)
    print("Falling back to normal search:")
    for document in retriever.invoke("how do I make saffron coffee")[:3]:
        print("  ", document.metadata["page"][:34], "|", document.metadata["section"])

## 9.3 · Compression and reranking — Session 4's two-step idea

**Step 1:** grab a lot of pieces quickly, so we don't miss the right one.
**Step 2:** something slower and smarter reads the question and each piece **together**, and keeps only what matters.

`ContextualCompressionRetriever` is a **wrapper**, not a single feature — swap what you put inside it and the behaviour changes completely:

| Inside | What it does | Cost |
|---|---|---|
| `LLMChainExtractor` | rewrites each piece down to the relevant sentences | one LLM call **per document** |
| `LLMChainFilter` | keeps or drops whole documents | one call per document |
| `EmbeddingsFilter` | drops documents below a similarity score | nearly free |

⚠️ **A bug you get for free by knowing this now:** `LLMChainExtractor` **rewrites the text**. If your citation marker lived inside the text, it's gone. Keep ids in `metadata` — which is exactly why Part 6 mattered.

In [ ]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

shortener = LLMChainExtractor.from_llm(llm)

retriever.top_k = 10
tidy_retriever = ContextualCompressionRetriever(
    base_compressor=shortener,
    base_retriever=retriever
)

question = "is turmeric coffee caffeine free"

before = retriever.invoke(question)
after = tidy_retriever.invoke(question)

before_size = 0
for document in before:
    before_size = before_size + len(document.page_content)

after_size = 0
for document in after:
    after_size = after_size + len(document.page_content)

print("before:", len(before), "pieces |", before_size, "characters")
print("after :", len(after), "pieces |", after_size, "characters")

print("\npage ids survived in metadata:")
for document in after:
    print("  ", document.metadata.get("page", "MISSING"))

retriever.top_k = 5


## 9.4 · Deduplication and diversity

Vector search happily returns ten near-identical chunks that all say the same thing. That wastes
the context window on one fact.

`DocumentCompressorPipeline` chains transformers together inside the same wrapper you just met:

| Transformer | Job |
|---|---|
| `EmbeddingsRedundantFilter` | drop near-duplicates, by vector similarity |
| `EmbeddingsFilter` | keep the top *k* most relevant, cheaply |
| `LongContextReorder` | move the strongest pieces to the **ends** of the list |

That last one is not a gimmick: models attend less reliably to the middle of a long context.

In [ ]:
from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
    EmbeddingsFilter
)
from langchain_community.document_transformers import (
    EmbeddingsRedundantFilter,
    LongContextReorder
)

tidy_pipeline = DocumentCompressorPipeline(transformers=[
    EmbeddingsRedundantFilter(embeddings=embeddings, similarity_threshold=0.92),
    EmbeddingsFilter(embeddings=embeddings, k=6),
    LongContextReorder()
])

diverse_retriever = ContextualCompressionRetriever(
    base_compressor=tidy_pipeline,
    base_retriever=retriever
)

retriever.top_k = 15
question = "ashwagandha coffee benefits"

before = retriever.invoke(question)
after = diverse_retriever.invoke(question)

print("before:", len(before), "pieces")
print("after :", len(after), "pieces")
for document in after:
    print("   ", document.metadata["page"][:36], "|", document.metadata["section"])

retriever.top_k = 5

### MMR — diversity at search time instead

**MMR** (maximal marginal relevance) attacks the same problem earlier: it picks results that are
relevant *and* different from each other, trading a little relevance for coverage.

It needs a plain vector store rather than the hybrid retriever.

In [ ]:
mmr_retriever = plain_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,              # how many we end up with
        "fetch_k": 30,       # how many we choose from
        "lambda_mult": 0.5   # 0 = maximum diversity, 1 = maximum relevance
    }
)

for document in mmr_retriever.invoke("ashwagandha coffee benefits"):
    print("   ", document.metadata.get("page", "?")[:36], "|", document.metadata.get("section"))

In [ ]:
# langchain-pinecone is not supported on Python 3.12, 3.13, or 3.14 because its current versions restrict Python support to <3.13 and depend on older simsimd packages (<4.0.0,>=3.6.3) that are incompatible or missing for modern Python runtimes.

## 9.5 · Cheap transforms — not everything needs a model

Multi-query and self-query both spend an LLM call. Plenty of useful query fixing is just Python:
deterministic, free, and instant.

In [ ]:
import re


def normalize_query(question):
    question = question.replace("celcius", "celsius")
    question = question.replace("miligrams", "milligrams")
    question = re.sub(r"\bdeg\b", "degrees", question)
    return question


messy = "brew at 90 deg celcius for ashwagandha coffe"
print("before:", messy)
print("after :", normalize_query(messy))

# translation is a plain chain — note there is no LLMChain in v1
translate_prompt = ChatPromptTemplate.from_messages([
    ("human", "Translate to English, keep the meaning. Reply with only the translation: {text}")
])
translate = translate_prompt | llm | StrOutputParser()

hindi = "क्या केसर कॉफी मीठी होती है?"
english = translate.invoke({"text": hindi})
print("\ntranslated:", english)
print("finds:", retriever.invoke(english)[0].metadata["page"])

## 9.6 · Fitting the context budget

Last step before prompting: cap what you send. A rough estimate is fine — roughly four characters
per token — and a greedy loop keeps the strongest pieces.

In [ ]:
def estimate_tokens(text):
    return max(1, len(text) // 4)


def fit_budget(documents, max_tokens=1200):
    kept = []
    total = 0
    for document in documents:
        needed = estimate_tokens(document.page_content)
        if total + needed > max_tokens:
            break
        kept.append(document)
        total = total + needed
    return kept, total


retriever.top_k = 15
candidates = retriever.invoke("ashwagandha coffee benefits")
final_documents, used = fit_budget(candidates, max_tokens=600)

print("retrieved:", len(candidates), "pieces")
print("kept     :", len(final_documents), "pieces |", used, "estimated tokens")
retriever.top_k = 5

> **Optional: the cross-encoder reranker from Session 4.**
> A cross-encoder scores the question and a passage **together**, which is far more accurate than
> comparing two independent vectors — and far too slow to run on the whole index, so it only ever
> touches a shortlist. It isn't a LangChain component; it's the same `sentence-transformers` model
> you used in Session 4, and it still works here:
>
> ```python
> from sentence_transformers import CrossEncoder
> reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
> scores = reranker.predict([(question, d.page_content) for d in candidates])
> ```
>
> The download is large and slow on a cold runtime, so we've left it out of the flow. Try it in
> your own time and compare the ordering against `LLMChainExtractor`.

---
# PART 10 · Rebuilding the Concierge

Everything is on the table. Time to build the thing Prasad promised the investors.

First the readable way — five steps, each on its own line with a name. These are the **same five steps as Session 5**; we've just swapped our hand-written pieces for LangChain ones.

Read it top to bottom. There is nothing clever here.


In [ ]:
def format_context(documents):
    lines = []
    for document in documents:
        lines.append("[" + document.metadata["page"] + "] " + document.page_content)
    return "\n\n".join(lines)


concierge_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the DilSeVector Concierge, a friendly coffee assistant.\n"
     "Answer ONLY from the CONTEXT below.\n"
     "Put the [page] at the end of every sentence that states a fact.\n"
     "If the CONTEXT does not answer the question, say you do not know. Never guess.\n"
     "Keep it under 80 words.\n\n"
     "CONTEXT:\n{context}"),
    ("human", "{question}")
])


def answer_question(question):
    documents = retriever.invoke(question)                 # 1 search
    context = format_context(documents)                    # 2 make it text
    filled = concierge_prompt.invoke({"context": context,
                                      "question": question})   # 3 fill the prompt
    reply = llm.invoke(filled)                             # 4 ask the model
    return reply.content                                   # 5 take the text out


print(answer_question("Is turmeric coffee caffeine free?"))

### The same thing as one chain

Steps 3, 4 and 5 clip together with the pipe. Steps 1 and 2 stay as ordinary Python, because we wrote `format_context` ourselves — **that's allowed.** You don't have to put everything in a chain.

In [ ]:
concierge_chain = concierge_prompt | llm | StrOutputParser()


def answer_question_short(question):
    documents = retriever.invoke(question)
    context = format_context(documents)
    return concierge_chain.invoke({"context": context, "question": question})


print(answer_question_short("How do I make masala coffee?"))

### The compact form you'll see online

Here is the whole thing — search included — as one expression. This is how LangChain examples are usually written.

The trick is the **dictionary at the front**. A dictionary of Runnables is itself a Runnable: it runs each value on the same input and hands back a dictionary. Those keys then fill the prompt's slots.

```
{"context": search and format,  "question": pass it through} | prompt | llm | parser
```

You do **not** need to write this way. But you should be able to read it.

In [ ]:
def search_and_format(question):
    documents = retriever.invoke(question)
    return format_context(documents)


full_chain = (
    {"context": RunnableLambda(search_and_format), "question": RunnablePassthrough()}
    | concierge_prompt
    | llm
    | StrOutputParser()
)

# note: the input is now just the question string, not a dictionary
print(full_chain.invoke("What spices go into masala coffee?"))

### The free superpowers

Now everything is one chain, and the whole thing streams and batches.

🎲 **Before you run it:** how many extra lines to make the answer appear word by word?

In [ ]:
print("STREAMING:\n")
for piece in full_chain.stream("How do I make masala coffee?"):
    print(piece, end="")

print("\n\nTWO QUESTIONS AT ONCE:")
for reply in full_chain.batch(["Is masala coffee sweet?", "What is jaggery coffee?"]):
    print("  -", reply[:90])

> **This is the real reason to use a framework.** Not shorter code — `.stream()`, `.batch()`, async, retries and tracing come attached to *every* Runnable, including chains you build yourself. In Session 5, adding streaming meant rewriting the model call.


### Enforcing citations

Asking nicely in the prompt is not a guarantee. Check, then repair — this is a LangChain-shaped
version of your Session 5 citation check.

Note which half is which: the **repair** is a chain, the **check** is ours. The framework has
nothing that verifies an answer.

In [ ]:
import re


def has_valid_citations(answer, documents):
    pages = []
    for document in documents:
        pages.append(document.metadata["page"])

    found = re.findall(r"\[([^\]]+)\]", answer or "")
    if not found:
        return False
    for page in found:
        if page not in pages:
            return False
    return True


repair_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the answer so every factual sentence ends with a [page] citation.\n"
     "Use ONLY these page ids: {allowed}. Do not invent sources. Do not change the facts."),
    ("human", "{answer}")
])
repair_chain = repair_prompt | llm | StrOutputParser()


question = "How do I make masala coffee?"
documents = retriever.invoke(question)
answer = concierge_chain.invoke({
    "context": format_context(documents),
    "question": question
})

print("cited properly?", has_valid_citations(answer, documents))

if not has_valid_citations(answer, documents):
    allowed = ", ".join(d.metadata["page"] for d in documents)
    answer = repair_chain.invoke({"allowed": allowed, "answer": answer})
    print("\nrepaired:\n", answer)

---
# PART 11 · Conversation memory

Session 5's lesson: **history is not memory.**

- **History** = the raw conversation so far. Grows fast. Goes into the prompt every turn.
- **Memory** = distilled facts about the person ("prefers low caffeine"). Searched like documents.

LangChain keeps history in its own prompt slot: `MessagesPlaceholder`. That's a reserved space for a *list of message objects*, kept structurally separate from the retrieved context.

That's the "User:" / "Assistant:" labelling we did by hand last week — now enforced by the message types.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the DilSeVector Concierge. Answer ONLY from the CONTEXT.\n"
     "Put the [page] after every fact. Keep it under 80 words.\n\n"
     "CONTEXT:\n{context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chat_chain = chat_prompt | llm | StrOutputParser()

history = []


def chat(question):
    # glue the last question on, so words like "it" still find the right page
    search_text = question
    if len(history) >= 2:
        search_text = history[-2].content + " " + question

    documents = retriever.invoke(search_text)
    context = format_context(documents)

    reply = chat_chain.invoke({"context": context, "history": history, "question": question})

    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=reply))
    return reply


print("👤 Tell me about ashwagandha coffee.")
print("🤖", chat("Tell me about ashwagandha coffee."))

print("\n👤 Compare it with turmeric coffee.")
print("🤖", chat("Compare it with turmeric coffee."))

print("\nMessages stored:", len(history))

### 11.2 · The managed form: `RunnableWithMessageHistory`

Now that you've seen the list grow by hand, here's the wrapper that maintains one **per session** —
which is what you need the moment there's more than one user.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

_sessions = {}


def get_session_history(session_id):
    if session_id not in _sessions:
        _sessions[session_id] = InMemoryChatMessageHistory()
    return _sessions[session_id]


chat_with_history = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="question",       # matches ("human", "{question}")
    history_messages_key="history"       # matches MessagesPlaceholder("history")
)

config = {"configurable": {"session_id": "user_001"}}

documents = retriever.invoke("What is beaten coffee?")
print(chat_with_history.invoke(
    {"context": format_context(documents), "question": "What is beaten coffee?"},
    config=config
))

print("\nmessages stored:", len(get_session_history("user_001").messages))

# to start fresh without touching the documents:
# _sessions["user_001"] = InMemoryChatMessageHistory()

> **Two things to keep in mind.**
> **1. Search every turn.** Memory gives continuity; searching gives grounding. If you answer from history alone, you're answering from the model's own earlier words — and mistakes compound.
> **2. History grows, the context window doesn't.** Real apps keep the last few turns, summarise older ones, and cap how much room history is allowed. LangChain has a `trim_messages` helper; LangGraph handles it properly with saved state.


---
# PART 12 · Self-Adaptive RAG

Everything so far ran the *same* pipeline for every question. Same `top_k`, same `alpha`, same
compression, whether the user asked "compare turmeric and saffron" or "what temperature?".

A **self-adaptive** RAG reads the question first and picks its own settings.

Notice what this actually is. We are not handing judgement back to the framework — we are writing
our own policy, in ordinary Python, over the knobs Part 8 and Part 9 taught us. The framework
supplies the knobs. The policy is ours.

| Question looks like | Policy |
|---|---|
| broad or comparative ("compare", "vs", "how", "steps") | wide recall, lean on **meaning**, compress |
| exact or numeric (digits, ratios, temperatures) | favour **exact words**, no compression — it clips numbers |
| anything else | balanced defaults |

In [ ]:
def plan_query(question):
    lowered = question.lower()

    broad_words = ["compare", "vs", "guide", "how", "steps", "difference"]
    is_broad = False
    for word in broad_words:
        if word in lowered:
            is_broad = True

    is_exact = False
    for character in lowered:
        if character.isdigit():
            is_exact = True

    if is_broad:
        return {"top_k": 12, "alpha": 0.8, "compress": True}
    if is_exact:
        return {"top_k": 8, "alpha": 0.2, "compress": False}
    return {"top_k": 8, "alpha": 0.5, "compress": True}


for question in ["Compare turmeric vs saffron coffee",
                 "What temperature for a 90 second brew?",
                 "What is jaggery coffee"]:
    print(plan_query(question), " <-", question)

### Look at which way `alpha` moves

Broad conceptual questions push alpha **up**, toward meaning. Exact numeric ones push it **down**,
toward words. That follows directly from Part 8 — and it's the reason we made you check the
library rather than trust a comment. Written backwards, this planner would still return
*something*, just quietly worse results, and nothing would ever tell you.

### One pass through the pipeline

In [ ]:
def answer_once(question, plan):
    retriever.top_k = plan["top_k"]
    retriever.alpha = plan["alpha"]

    if plan["compress"]:
        source = ContextualCompressionRetriever(
            base_compressor=shortener,
            base_retriever=retriever
        )
    else:
        source = retriever

    documents = source.invoke(question)
    context = format_context(documents)
    answer = concierge_chain.invoke({"context": context, "question": question})
    return answer, documents

### Check the answer, and retry wider if it's weak

A second model call grades the first one. If confidence is low, we widen recall **and turn
compression off** — because a low-confidence answer usually means we threw away the sentence we
needed.

In [ ]:
verify_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict verifier. Compare the ANSWER to the CONTEXT.\n"
     'Reply with ONLY JSON: {{"supported": true, "confidence": 0.0, "problems": []}}\n'
     "confidence is 0.0 to 1.0, judged from the CONTEXT alone."),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}")
])

verify_chain = verify_prompt | llm | JsonOutputParser()


def adaptive_answer(question):
    plan = plan_query(question)
    answer, documents = answer_once(question, plan)

    # a model asked for JSON does not always return JSON — never let the
    # verifier take down the answer we already have
    try:
        verdict = verify_chain.invoke({
            "context": format_context(documents),
            "question": question,
            "answer": answer
        })
    except Exception as problem:
        print("   (verifier did not return JSON:", type(problem).__name__, "- keeping the answer)")
        verdict = {"supported": True, "confidence": 1.0, "problems": []}

    confidence = verdict.get("confidence", 0.0)
    unknown = "do not know" in answer.lower()

    if confidence < 0.5 or unknown:
        print("   (low confidence -> retrying wider)")
        wider = {"top_k": plan["top_k"] + 8, "alpha": 0.5, "compress": False}
        answer, documents = answer_once(question, wider)
        plan = wider

    retriever.top_k = 5      # always put the knobs back
    retriever.alpha = 0.5
    return answer, plan, verdict


final_answer, used_plan, verdict = adaptive_answer(
    "Compare turmeric coffee and saffron coffee"
)
print("PLAN USED:", used_plan)
print("CONFIDENCE:", verdict.get("confidence"))
print("\nANSWER:\n", final_answer)

> **Two honest notes.**
> **1.** The verifier costs a second LLM call, so hard questions cost roughly double and easy ones
> cost nothing extra. That's the point — spend tokens where they matter.
> **2.** The verifier is **not** one of your Session 5 checks. It is a language model grading
> another language model. Useful, but it is not evidence. Part 14 comes back to this.

---
# PART 13 · Seeing inside your chain

The honest cost of a framework: when something breaks, the error comes from inside someone else's code. So learn the tools that let you look inside.

**1. Run each piece separately.** The best debugging tool you have — every Runnable has `.invoke()`, so you can test any step alone.
**2. Print the filled prompt.** Most bad answers are bad prompts.
**3. LangSmith.** Two environment variables and every step, prompt, token count and timing is recorded in a web view. This is the argument for LangChain that has nothing to do with line count.

In [ ]:
# 1. test any single step on its own
documents = retriever.invoke("is masala coffee sweet")
print("step 1 gave", len(documents), "documents")

# 2. look at the exact prompt the model received
context = format_context(documents)
filled = concierge_prompt.invoke({"context": context, "question": "Is masala coffee sweet?"})
print("\n--- what the model actually sees ---")
print(filled.messages[0].content[:400], "...")
print("\n--- the human turn ---")
print(filled.messages[1].content)

# 3. LangSmith (uncomment and add a key to try it)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "your-langsmith-key"
# full_chain.invoke("Is masala coffee sweet?")   # then open smith.langchain.com

---
# PART 14 · What LangChain gave us, and what it left us

| Session 5, by hand | Session 6, LangChain | **You still decide** |
|---|---|---|
| our own `search()` | a retriever, and it's a Runnable | **alpha, top_k** |
| reading pages with BeautifulSoup | still ours | which sections matter |
| our own chunking | `RecursiveCharacterTextSplitter` | **chunk_size, overlap** |
| calling the embedder in a loop | an embeddings object | which model (and re-index if it changes) |
| prompt as one f-string | `ChatPromptTemplate`, system vs human | **every word of it** |
| `reply.text` by hand | `StrOutputParser`, `JsonOutputParser` | — |
| ids in a parallel dictionary | `Document.metadata` carries them | what to store |
| our hand-written router | `SelfQueryRetriever`, JSON parsing | when to trust it |
| our shortlist code | `ContextualCompressionRetriever` | what goes inside the wrapper |
| our own dedup pass | `EmbeddingsRedundantFilter`, MMR | the similarity threshold |
| our adaptive branching | **nothing — we wrote the policy in Part 12** | every rule in it |
| "User:" / "Assistant:" labels | `MessagesPlaceholder` | what to keep, when to summarise |
| five steps wired by hand | `prompt \| llm \| parser` | — |
| streaming: not attempted | free on every Runnable | — |
| **our four checks (E F C R)** | **nothing — there is no such tool** | **all of it** |

### Look hard at that last row.

LangChain has components for searching, prompting, calling models, routing, reranking, compressing and remembering. It has **nothing** that checks whether an answer is actually true.

The four checks you wrote in Session 5 have no replacement here. They also need no changes — they only need the answer text and the documents, both of which this notebook produces.

> *"Our Session 5 Concierge was a few hundred lines. This one is a few dozen. But every setting in that right-hand column — alpha, top_k, chunk size, the prompt, the checks — is something you can only choose because you built it by hand first. LangChain made us fast. Session 5 made us right. A framework in the hands of someone without judgement just builds broken things faster."*

---

### Back at the café

Prasad gets his three apps. Each one is this notebook with a different index behind the retriever and a different persona in the prompt — the chain itself doesn't change. That's what today bought us.

And into every one of them, the four checks from Session 5 travel along, hand-carried, because no framework will carry them for us.

---

## Homework

1. **Prove the last row.** Copy `find_citations` and `check_citations_real` from your Session 5 notebook into this one and run them on an answer from `answer_question()`. Nothing should need changing.

2. **Swap a provider.** Change the string in `init_chat_model` to a different model and re-run Part 10. Count how many other lines you had to touch.

3. **Find alpha's breaking point.** Find one question that only works at `alpha = 0.0` and one that only works at `alpha = 1.0`. Bring both next session.

4. **Break the compressor.** Put a citation marker inside `page_content` instead of metadata, run it through `ContextualCompressionRetriever`, and see what happens to it.

5. **Rebuild fusion.** Reimplement Session 5's score normalisation and weighted fusion as a function that takes two ranked lists and returns one.

6. **Tune the planner.** Part 12's `plan_query` uses crude keyword rules. Find a question it plans badly for, and fix the rule.

## Quick reference

| I want to… | Use |
|---|---|
| call any model | `init_chat_model("provider:model")` |
| build a prompt with roles | `ChatPromptTemplate.from_messages([...])` |
| add conversation history | `MessagesPlaceholder("history")` |
| get plain text out | `StrOutputParser()` |
| get a dictionary out | `JsonOutputParser()` |
| cut long text up | `RecursiveCharacterTextSplitter(chunk_size=…, chunk_overlap=…)` |
| carry text plus its id | `Document(page_content=…, metadata=…)` |
| search several wordings | `MultiQueryRetriever.from_llm(...)` |
| let the model write a filter | `SelfQueryRetriever.from_llm(...)` |
| shrink or rerank results | `ContextualCompressionRetriever(...)` |
| drop near-duplicate results | `EmbeddingsRedundantFilter(...)` in a `DocumentCompressorPipeline` |
| get varied results | `as_retriever(search_type="mmr", ...)` |
| keep history per user | `RunnableWithMessageHistory(...)` |
| decompose a multi-part question | `MultiQueryRetriever.from_llm(..., prompt=decomp_prompt)` |
| join steps | `step_one \| step_two` |
| pass a value through untouched | `RunnablePassthrough()` |
| turn my function into a step | `RunnableLambda(my_function)` |


---
# Appendix A · Migrating old LangChain code

Every import on the left raises `ModuleNotFoundError` on a clean v1 install. If a student is
following a blog post or an older handout, this table is the fix.

| Old (pre-v1) | New |
|---|---|
| `langchain.prompts` | `langchain_core.prompts` |
| `langchain.schema` | `langchain_core.messages` |
| `langchain.text_splitter` | `langchain_text_splitters` |
| `langchain.chains.LLMChain` | *(removed)* — use `prompt` \| `llm` \| `parser` |
| `langchain.retrievers.MultiQueryRetriever` | `langchain_classic.retrievers.multi_query` |
| `langchain.retrievers.self_query.base` | `langchain_classic.retrievers.self_query.base` |
| `langchain.retrievers.ContextualCompressionRetriever` | `langchain_classic.retrievers.contextual_compression` |
| `langchain.retrievers.document_compressors` | `langchain_classic.retrievers.document_compressors` |
| `langchain.chains.query_constructor.schema.AttributeInfo` | `langchain_classic.chains.query_constructor.schema` |
| `langchain_community.vectorstores.Pinecone` | `langchain_pinecone.PineconeVectorStore` |
| `langchain_community.chat_message_histories.ChatMessageHistory` | `langchain_core.chat_history.InMemoryChatMessageHistory` |

### Methods that changed

| Old | New |
|---|---|
| `retriever.get_relevant_documents(q)` | `retriever.invoke(q)` |
| `chain.run({...})` | `chain.invoke({...})` |
| `LLMChain(...).run(x)` | `(prompt` \| `llm` \| `StrOutputParser()).invoke(x)` |

⚠️ `get_relevant_documents` is **removed**, not deprecated. Calling it raises `AttributeError`.

Still in `langchain_community` — works today, but the package is being sunset, so expect these to
move: `PineconeHybridSearchRetriever`, `HuggingFaceEmbeddings`, `EmbeddingsRedundantFilter`,
`LongContextReorder`, the document loaders.

---
# Appendix B · When something breaks

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'langchain.retrievers'` | pre-v1 import | see Appendix A |
| `AttributeError: … has no attribute 'get_relevant_documents'` | removed in v1 | use `.invoke()` |
| Pinecone dimension mismatch | mixing a 384-dim model with our 768-dim index | one index, one embedding model |
| `SelfQueryRetriever` fails on import | `lark` missing | it's in the install cell — re-run it |
| Self-query returns zero documents | the model invented a filter value that doesn't exist | fall back to the plain retriever |
| Citations vanished after compression | `LLMChainExtractor` rewrote `page_content` | keep ids in `metadata` |
| `KeyError` on a literal `{` in a prompt | single braces are template slots | double them: `{{` and `}}` |
| Answers changed after Part 9 | `top_k` or `alpha` left at an experiment's value | reset them after experiments |
| First run is very slow | cold runtime, model downloads | run the install cell before you start |